# 💰 Insurance Premium Prediction
### Neural Network Regression with Feature Scaling & Normalization

---

## Project Overview
Predict **medical insurance charges** for patients based on demographic and lifestyle features using a deep neural network regression model.  
This project demonstrates **feature engineering**, **one-hot encoding**, **normalization**, and **neural network regression** on a real-world tabular dataset.

**Dataset:** [Kaggle Medical Insurance Dataset](https://www.kaggle.com/datasets/mirichoi0218/insurance)  
**Task:** Regression — Predict Insurance Charges (USD)  
**Input Features:** age, sex, bmi, children, smoker, region  
**Target Variable:** charges

---

## Table of Contents
1. [Import Libraries](#1)
2. [Load & Explore Data](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Feature Engineering & Encoding](#4)
5. [Feature Scaling](#5)
6. [Build Neural Network](#6)
7. [Train the Model](#7)
8. [Evaluate & Visualize](#8)
9. [Make Predictions](#9)

## 1. Import Libraries

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('TensorFlow version:', tf.__version__)
tf.random.set_seed(42)
np.random.seed(42)

## 2. Load & Explore Data

In [ ]:
# Download dataset
# Option 1: Kaggle API
# !pip install kagglehub --quiet
# import kagglehub; path = kagglehub.dataset_download('mirichoi0218/insurance')

# Option 2: Load local file (download from Kaggle and place in same directory)
df = pd.read_csv('insurance.csv')

print('Shape:', df.shape)
print('\nColumn types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Statistical Summary ===')
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Distribution of charges
axes[0,0].hist(df['charges'], bins=40, color='steelblue', edgecolor='white')
axes[0,0].set_title('Distribution of Insurance Charges')
axes[0,0].set_xlabel('Charges (USD)')

# Charges by smoker
df.boxplot(column='charges', by='smoker', ax=axes[0,1])
axes[0,1].set_title('Charges by Smoker Status')

# Charges by region
df.boxplot(column='charges', by='region', ax=axes[0,2])
axes[0,2].set_title('Charges by Region')
axes[0,2].tick_params(axis='x', rotation=15)

# Age vs Charges
colors = df['smoker'].map({'yes':'red', 'no':'steelblue'})
axes[1,0].scatter(df['age'], df['charges'], c=colors, alpha=0.5)
axes[1,0].set_title('Age vs Charges (red=smoker)')
axes[1,0].set_xlabel('Age')
axes[1,0].set_ylabel('Charges')

# BMI vs Charges
axes[1,1].scatter(df['bmi'], df['charges'], c=colors, alpha=0.5)
axes[1,1].set_title('BMI vs Charges (red=smoker)')
axes[1,1].set_xlabel('BMI')

# Correlation heatmap (numeric only)
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', ax=axes[1,2], fmt='.2f')
axes[1,2].set_title('Correlation Heatmap')

plt.suptitle('Insurance Dataset — Exploratory Analysis', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Feature Engineering & Encoding

In [ ]:
# One-hot encode all categorical features
insurance_encoded = pd.get_dummies(df, drop_first=False)

print('Columns after encoding:')
print(list(insurance_encoded.columns))
print('\nShape:', insurance_encoded.shape)
insurance_encoded.head()

In [ ]:
# Define features and target
X = insurance_encoded.drop('charges', axis=1)
y = insurance_encoded['charges']

print('Features shape:', X.shape)
print('Target range:  $%.0f — $%.0f' % (y.min(), y.max()))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'\nTrain size: {len(X_train)}, Test size: {len(X_test)}')

## 5. Feature Scaling

In [ ]:
# Standardization: mean=0, std=1 — best for neural networks
scaler   = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Also scale target variable for smoother gradients
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled  = y_scaler.transform(y_test.values.reshape(-1, 1)).ravel()

print('X_train scaled shape:', X_train_scaled.shape)
print('Scaling complete!')

## 6. Build Neural Network

In [ ]:
def build_regression_model(input_dim):
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1)  # Linear output — regression
    ], name='InsuranceRegressor')

    model.compile(
        loss='mse',
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        metrics=['mae']
    )
    return model

model = build_regression_model(X_train_scaled.shape[1])
model.summary()

## 7. Train the Model

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_mae', patience=15, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_mae', factor=0.5, patience=7, min_lr=1e-6, verbose=1)
]

history = model.fit(
    X_train_scaled, y_train_scaled,
    epochs=200,
    batch_size=32,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)

print('Training complete!')

## 8. Evaluate & Visualize

In [ ]:
# Predict and inverse-transform to get real USD values
y_pred_scaled = model.predict(X_test_scaled).ravel()
y_pred        = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'MAE:  ${mae:,.0f}')
print(f'RMSE: ${rmse:,.0f}')
print(f'R²:   {r2:.4f} ({r2*100:.1f}% variance explained)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Training loss curve
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Training Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Predicted vs Actual
axes[1].scatter(y_test, y_pred, alpha=0.5, color='steelblue')
perfect_line = np.linspace(y_test.min(), y_test.max(), 100)
axes[1].plot(perfect_line, perfect_line, 'r--', label='Perfect prediction')
axes[1].set_title(f'Predicted vs Actual (R²={r2:.3f})')
axes[1].set_xlabel('Actual Charges ($)')
axes[1].set_ylabel('Predicted Charges ($)')
axes[1].legend()

# Residuals
residuals = y_test.values - y_pred
axes[2].hist(residuals, bins=40, color='steelblue', edgecolor='white')
axes[2].axvline(0, color='red', linestyle='--')
axes[2].set_title('Residuals Distribution')
axes[2].set_xlabel('Residual (Actual - Predicted)')

plt.tight_layout()
plt.show()

## 9. Make Predictions on New Patients

In [ ]:
def predict_insurance_charge(age, sex, bmi, children, smoker, region):
    """
    Predict medical insurance charge for a new patient.
    sex: 'male'/'female' | smoker: 'yes'/'no'
    region: 'northeast'/'northwest'/'southeast'/'southwest'
    """
    sample = pd.DataFrame({'age':[age], 'sex':[sex], 'bmi':[bmi],
                            'children':[children], 'smoker':[smoker], 'region':[region]})
    sample_encoded = pd.get_dummies(sample, drop_first=False)

    # Align columns with training data
    sample_encoded = sample_encoded.reindex(columns=X.columns, fill_value=0)
    sample_scaled  = scaler.transform(sample_encoded)

    pred_scaled = model.predict(sample_scaled, verbose=0)
    pred_charge = y_scaler.inverse_transform(pred_scaled)[0][0]
    print(f'Estimated insurance charge: ${pred_charge:,.0f}')
    return pred_charge

# Example predictions
print('=== Example Predictions ===')
predict_insurance_charge(age=25, sex='female', bmi=22.0, children=0, smoker='no',  region='northeast')
predict_insurance_charge(age=45, sex='male',   bmi=30.0, children=2, smoker='yes', region='southeast')
predict_insurance_charge(age=60, sex='female', bmi=28.5, children=3, smoker='no',  region='southwest')